# Signal Implementation: Cross-Sectional Quality-of-Momentum

This notebook implements the **most feasible signal** from the previous research pass:

- **Signal**: Cross-sectional quality-of-momentum
- **Prior feasibility rating**: **Very High**

## Step-by-step execution contract
We do not proceed unless each checkpoint passes:

1. Environment + path guardrails
2. Data availability checks
3. Schema checks
4. Score construction
5. Signal construction
6. Backtest execution
7. Explainability diagnostics

Every step has explicit validation and will raise a clear error if assumptions fail.


In [ ]:
# Step 1 — Setup imports, robust paths, and step checkpoints
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("tab10")

pd.set_option("display.max_columns", 200)
pd.set_option("display.precision", 4)


def resolve_first_existing(paths: list[Path], fallback: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return fallback


CWD = Path.cwd().resolve()
CANDIDATES_ROOT = [CWD, CWD.parent]

for root in CANDIDATES_ROOT:
    if (root / "config" / "settings.py").exists():
        PROJECT_ROOT = root
        break
else:
    PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = resolve_first_existing(
    [root / "data" / "factors" for root in CANDIDATES_ROOT],
    PROJECT_ROOT / "data" / "factors",
)
DUCKDB_PATH = resolve_first_existing(
    [root / "data" / "factors" / "factors.duckdb" for root in CANDIDATES_ROOT],
    DATA_DIR / "factors.duckdb",
)
FACTOR_PATH = DATA_DIR / "factors_price.parquet"
PRICE_PATH = DATA_DIR / "prices.parquet"

STEP_STATUS: dict[str, bool] = {}


def confirm_step(step_id: str, condition: bool, success_message: str, failure_message: str) -> None:
    if not condition:
        raise RuntimeError(f"❌ {step_id} failed: {failure_message}")
    STEP_STATUS[step_id] = True
    print(f"✅ {step_id} confirmed: {success_message}")


def require_steps(required_step_ids: list[str]) -> None:
    missing = [step_id for step_id in required_step_ids if step_id not in STEP_STATUS]
    if missing:
        raise RuntimeError(
            "❌ Step order violation. Execute and confirm these steps first: "
            + ", ".join(missing)
        )


print(f"Project root   : {PROJECT_ROOT}")
print(f"Data directory : {DATA_DIR}")
print(f"DuckDB path    : {DUCKDB_PATH}")
print(f"Factor parquet : {FACTOR_PATH}")
print(f"Prices parquet : {PRICE_PATH}")
print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"DuckDB exists  : {DUCKDB_PATH.exists()}")
if not DUCKDB_PATH.exists():
    print("⚠️ DuckDB file missing. This implementation uses parquet directly, so it can still run.")

confirm_step(
    "step_1_setup",
    PROJECT_ROOT.exists() and (PROJECT_ROOT / "config" / "settings.py").exists(),
    "Project root and config were resolved robustly.",
    "Could not resolve project root. Start Jupyter in repo root or notebooks/ and ensure config/settings.py exists.",
)


In [ ]:
# Step 2 — Data availability checkpoint
require_steps(["step_1_setup"])

candidates = [
    PRICE_PATH,
    FACTOR_PATH,
    DATA_DIR / "factors_all.parquet",
    DATA_DIR / "macro.parquet",
    DATA_DIR / "macro_z.parquet",
    DUCKDB_PATH,
]

inventory_rows = []
for path in candidates:
    inventory_rows.append(
        {
            "path": str(path.relative_to(PROJECT_ROOT)) if path.exists() else str(path),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / (1024**2), 2) if path.exists() else np.nan,
        }
    )

inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df)

required_files_exist = FACTOR_PATH.exists() and PRICE_PATH.exists()
confirm_step(
    "step_2_data_availability",
    required_files_exist,
    "Required parquet files are present.",
    "Required files missing: expected data/factors/factors_price.parquet and data/factors/prices.parquet.",
)


## Step 3 — Load data and confirm schema

At this checkpoint we verify that the implementation inputs are structurally sound:

- Factor panel has MultiIndex (`date`, `symbol`)
- Required columns exist: `mom_12_1`, `vol_60d`, `beta_60d`
- Prices table aligns to symbols and time range for backtesting

Only after these validations pass do we compute the score.


require_steps(["step_2_data_availability"])

factors = pd.read_parquet(FACTOR_PATH)
prices = pd.read_parquet(PRICE_PATH)

print(f"factors shape: {factors.shape}")
print(f"prices  shape: {prices.shape}")

required_factor_columns = {"mom_12_1", "vol_60d", "beta_60d"}
missing_cols = sorted(required_factor_columns - set(factors.columns))

# Index validations
has_multiindex = isinstance(factors.index, pd.MultiIndex)
index_names_ok = has_multiindex and set(factors.index.names) >= {"date", "symbol"}

confirm_step(
    "step_3_schema",
    has_multiindex and index_names_ok and len(missing_cols) == 0,
    "Factor panel index and required columns are valid.",
    (
        f"Schema mismatch. MultiIndex/date-symbol required and columns missing={missing_cols}. "
        "Regenerate factors pipeline artifacts before continuing."
    ),
)

# Normalize time index for alignment checks
if not isinstance(prices.index, pd.DatetimeIndex):
    prices.index = pd.to_datetime(prices.index)

factor_dates = pd.to_datetime(factors.index.get_level_values("date"))
factor_symbols = pd.Index(factors.index.get_level_values("symbol").unique())
price_symbols = pd.Index(prices.columns)

symbol_overlap = factor_symbols.intersection(price_symbols)
confirm_step(
    "step_3b_alignment",
    len(symbol_overlap) >= 20,
    f"Sufficient symbol overlap for backtesting ({len(symbol_overlap)} symbols).",
    "Not enough symbol overlap between factors and prices; check symbol mapping and data refresh.",
)

print(f"Date range factors: {factor_dates.min()} -> {factor_dates.max()}")
print(f"Date range prices : {prices.index.min()} -> {prices.index.max()}")
print(f"Overlapping symbols: {len(symbol_overlap)}")


## Step 4 — Construct an explainable quality-of-momentum score

We define three interpretable components, each scaled into `[0, 1]` cross-sectionally by date:

- `mom_rank`: higher 12-1 momentum is better.
- `low_vol_rank`: lower 60d volatility is better.
- `beta_stability_rank`: beta closer to `1.0` is better.

Final score (equal-weight baseline):

`qom_score = (mom_rank + low_vol_rank + beta_stability_rank) / 3`

This decomposition makes attribution explicit and avoids opaque transformations.


require_steps(["step_3_schema", "step_3b_alignment"])

score_df = factors[["mom_12_1", "vol_60d", "beta_60d"]].copy()

# Keep finite observations only for robust ranking
finite_mask = np.isfinite(score_df["mom_12_1"]) & np.isfinite(score_df["vol_60d"]) & np.isfinite(score_df["beta_60d"])
score_df = score_df.loc[finite_mask].copy()

# Cross-sectional percentile ranks per date (0..1)
score_df["mom_rank"] = score_df.groupby(level="date")["mom_12_1"].rank(pct=True, ascending=True)
score_df["low_vol_rank"] = score_df.groupby(level="date")["vol_60d"].rank(pct=True, ascending=False)
score_df["beta_distance"] = (score_df["beta_60d"] - 1.0).abs()
score_df["beta_stability_rank"] = score_df.groupby(level="date")["beta_distance"].rank(pct=True, ascending=False)

score_df["qom_score"] = (
    score_df["mom_rank"] + score_df["low_vol_rank"] + score_df["beta_stability_rank"]
) / 3.0

component_cols = ["mom_rank", "low_vol_rank", "beta_stability_rank", "qom_score"]
within_bounds = score_df[component_cols].ge(0).all().all() and score_df[component_cols].le(1).all().all()

confirm_step(
    "step_4_score_construction",
    within_bounds and score_df["qom_score"].notna().any(),
    "Score components and final score were built and validated.",
    "Score construction failed bounds/non-null checks; inspect factor distributions for problematic dates.",
)

score_df[component_cols].describe().T


## Step 5 — Convert score into long/short signals

Signal logic is transparent and date-local:

- Long = top 20% by `qom_score`
- Short = bottom 20% by `qom_score`
- Minimum cross-sectional breadth: at least 20 names/date

This follows the existing project backtest conventions.


require_steps(["step_4_score_construction"])

from core.backtest.portfolio import calculate_portfolio_returns, create_signals_from_factor
from core.metrics.performance import calculate_cumulative_returns, calculate_performance_metrics

TOP_PCT = 0.20
BOTTOM_PCT = 0.20
MIN_STOCKS = 20
TRANSACTION_COST = 0.001
REBALANCE_FREQ = "ME"

signal_input = factors.copy()
signal_input["qom_score"] = score_df["qom_score"]

signals = create_signals_from_factor(
    factors_df=signal_input,
    factor_col="qom_score",
    top_pct=TOP_PCT,
    bottom_pct=BOTTOM_PCT,
    min_stocks=MIN_STOCKS,
)

signal_counts = signals["signal"].value_counts().to_dict()
non_zero_signals = int((signals["signal"] != 0).sum())

confirm_step(
    "step_5_signal_generation",
    non_zero_signals > 0 and 1 in signal_counts and -1 in signal_counts,
    f"Signals generated with both long and short legs. Non-zero rows={non_zero_signals:,}.",
    "Signal generation produced empty or one-sided positions. Check score coverage and min_stocks threshold.",
)

pd.Series(signal_counts, name="count").to_frame()


## Step 6 — Backtest the signal and verify portfolio sanity

We now run the existing project backtester with monthly rebalancing and transaction costs.

Sanity confirmations:

- Net return series exists and is non-empty
- There are actual invested days (`n_long + n_short > 0`)
- Output metrics are finite

In [ ]:
require_steps(["step_5_signal_generation"])

portfolio = calculate_portfolio_returns(
    signals=signals,
    prices=prices,
    rebalance_freq=REBALANCE_FREQ,
    transaction_cost=TRANSACTION_COST,
)

net_returns = portfolio["net_return"].dropna()
active_days = int(((portfolio["n_long"] + portfolio["n_short"]) > 0).sum())

metrics = calculate_performance_metrics(net_returns)
metrics_df = pd.DataFrame([metrics]).T.rename(columns={0: "value"})

is_finite_metrics = np.isfinite(pd.Series(metrics, dtype="float64").replace([np.inf, -np.inf], np.nan)).all()

confirm_step(
    "step_6_backtest",
    len(net_returns) > 0 and active_days > 0 and bool(is_finite_metrics),
    f"Backtest completed with {len(net_returns):,} return observations and {active_days:,} active days.",
    "Backtest output failed sanity checks (empty returns, no positions, or non-finite metrics).",
)

display(metrics_df)

## Step 7 — Explainability and justification diagnostics

We explicitly justify *why* the strategy holds names by linking score components to outcomes:

1. **Attribution by component**: contribution weights are transparent (equal thirds).
2. **Top-vs-bottom spread**: compare forward returns of top and bottom score buckets.
3. **Visual diagnostics**: cumulative returns and component distribution checks.

This is intentionally interpretable-first, not complexity-first.

In [ ]:
require_steps(["step_6_backtest"])

# Build one-period-ahead cross-sectional return label for explainability checks
returns = prices.pct_change().shift(-1)
returns_long = returns.stack().rename("fwd_1d_ret")
returns_long.index = returns_long.index.set_names(["date", "symbol"])

explain_df = score_df[["mom_rank", "low_vol_rank", "beta_stability_rank", "qom_score"]].join(
    returns_long,
    how="inner",
)

# Per-date bucket assignment (quintiles) for interpretable spread testing
explain_df["qom_bucket"] = explain_df.groupby(level="date")["qom_score"].transform(
    lambda s: pd.qcut(s, q=5, labels=False, duplicates="drop") if s.notna().sum() >= 5 else np.nan
)

bucket_returns = explain_df.dropna(subset=["qom_bucket", "fwd_1d_ret"]).groupby(
    [explain_df.dropna(subset=["qom_bucket", "fwd_1d_ret"]).index.get_level_values("date"), "qom_bucket"]
)["fwd_1d_ret"].mean().unstack("qom_bucket")

# Highest minus lowest bucket spread (when both available)
if 0 in bucket_returns.columns and bucket_returns.columns.max() in bucket_returns.columns:
    low_col = 0
    high_col = int(bucket_returns.columns.max())
    spread = (bucket_returns[high_col] - bucket_returns[low_col]).dropna()
else:
    spread = pd.Series(dtype="float64")

confirm_step(
    "step_7_explainability",
    len(explain_df) > 0 and len(spread) > 0,
    f"Explainability diagnostics created with {len(spread):,} daily spread observations.",
    "Insufficient data to compute explainability spread diagnostics.",
)

print("Component means:")
display(explain_df[["mom_rank", "low_vol_rank", "beta_stability_rank", "qom_score"]].mean().to_frame("mean"))

print("Forward-return spread summary (top bucket - bottom bucket):")
display(spread.describe().to_frame("spread_stats"))

In [ ]:
require_steps(["step_7_explainability"])

# Visual confirmation: strategy equity and explainability spread
cum_returns = calculate_cumulative_returns(net_returns)
spread_cum = (1.0 + spread.fillna(0.0)).cumprod() if len(spread) else pd.Series(dtype="float64")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

axes[0].plot(cum_returns.index, cum_returns.values, linewidth=1.2, label="QoM long/short")
axes[0].set_title("Implemented Signal: Cumulative Net Return")
axes[0].set_ylabel("Wealth index")
axes[0].legend(loc="upper left")

if len(spread_cum):
    axes[1].plot(spread_cum.index, spread_cum.values, linewidth=1.2, color="purple", label="Top-bottom bucket spread")
    axes[1].legend(loc="upper left")
axes[1].set_title("Explainability Check: Cumulative Top-minus-Bottom Score Spread")
axes[1].set_ylabel("Wealth index")

fig.tight_layout()
plt.show()

In [ ]:
# Final checkpoint summary
audit_steps = [
    "step_1_setup",
    "step_2_data_availability",
    "step_3_schema",
    "step_3b_alignment",
    "step_4_score_construction",
    "step_5_signal_generation",
    "step_6_backtest",
    "step_7_explainability",
]

summary = pd.DataFrame(
    {
        "step": audit_steps,
        "confirmed": [step in STEP_STATUS for step in audit_steps],
    }
)
display(summary)

confirm_step(
    "step_final_audit",
    summary["confirmed"].all(),
    "All implementation checkpoints passed in sequence.",
    "One or more checkpoints are missing. Re-run cells in order.",
)

print("Notebook implementation is complete and validated.")